Installing libraries:

transformers, accelerate, bitsandbytes, torch: for loading and running deep seek models efficintly.

datasets: To load financial news data.

gradio: for building UI

In [1]:
print("Installing libraries...")
!pip install -q transformers accelerate bitsandbytes>=0.46.1 datasets torch gradio
print("successfully installed!")
print("Please restart the runtime after this installation to ensure bitsandbytes is properly loaded.")

Installing libraries...
successfully installed!
Please restart the runtime after this installation to ensure bitsandbytes is properly loaded.


In [2]:
#hugging face login
import os
from huggingface_hub import login, notebook_login
print("logging in hugging face ...")
notebook_login()
print("successfully logged in!")

logging in hugging face ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


successfully logged in!


In [3]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from datasets import load_dataset #import the dataset loading function
import gradio as gr
from IPython.display import display, Markdown
import random #to pick random news item

#check GPU availability
if torch.cuda.is_available():
  print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
  torch.set_default_device("cuda")
  print("pytorch default device set to cuda(GPU).")
else:
  print("No GPU detected. Running on CPU")

GPU Detected: Tesla T4
pytorch default device set to cuda(GPU).


In [4]:
#helper function for markdown display
def print_markdown(text):
  """Displays text as markdown"""
  display(Markdown(text))

**TASK: Loading financial news datasets from hugging face**


1.   Use load_dataset("dataset_name") to download and cache the dataset
2.   inspect the dataset structure and features


1.  select relavant columns(title and description) for analysis
2.  combine the title and description into a single text input to the llm





In [5]:
dataset_id = "mohit9999/all_news_finance_sm_1h2023_custom"
print(f"Loading dataset:{dataset_id}")
#load the dataset
#we need only 'train' split, specify split = 'train' if needed
#The datatype of news_dataset is datasets.Datasets
news_dataset = load_dataset(dataset_id, split = 'train')
print("dataset loaded successfully")

Loading dataset:mohit9999/all_news_finance_sm_1h2023_custom


README.md:   0%|          | 0.00/349 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5062 [00:00<?, ? examples/s]

dataset loaded successfully


In [6]:
#inspecting the dataset
news_dataset

Dataset({
    features: ['instruction', 'output', 'text'],
    num_rows: 5062
})

In [7]:
#displaying features
print("\n Dataset Features:")
print(news_dataset.features)


 Dataset Features:
{'instruction': Value('string'), 'output': Value('string'), 'text': Value('string')}


In [8]:
#view datasets in pandas dataframe
news_dataset.to_pandas()

,instruction,output,text
0,"Dow drops 400 points, turns negative for the y...",Regional banks led the broader market lower as...,Below is an instruction that describes a task....
1,Bitcoin drops with stocks as analyst warns of ...,Bitcoin dips as the U.S. banking crisis engulf...,Below is an instruction that describes a task....
2,Bitcoin Price Analysis: 29370 Tested After Su...,Bitcoin (BTC/USD) sought to add to recent gain...,Below is an instruction that describes a task....
3,"Bitcoin Is 75% To Halving, Here's How Past Cyc...",The current Bitcoin cycle is now 75% on the wa...,Below is an instruction that describes a task....
4,"Iron Mountain FFO of $0.71 beats by $0.03, rev...",Iron Mountain press release (IRM): Q1 FFO of...,Below is an instruction that describes a task....
...,...,...,...
5057,"Ethereum Merge can trigger high volatility, Bi...",BitMEX CEO Alexander Höptner said that their t...,Below is an instruction that describes a task....
5058,EXPLAINER: Is COVID-19 winding down? Scientist...,Is the coronavirus on its way out? You might t...,Below is an instruction that describes a task....
5059,How Bitcoin Educates The World About Finance,Bitcoin brings financial education to the fore...,Below is an instruction that describes a task....
5060,Crypto Hedge Fund Three Arrows Capital Wallet ...,The removal of liquidity was the first activit...,Below is an instruction that describes a task....


In [9]:
#preparing data for llm
#combine title and description for input text
def combine_news_text(example):
  title = example.get("instruction","") or ""
  description = example.get("output","") or ""
  return {"full_text": f"Title:{title}\nDescription{description}"}

In [10]:
#applying the function
#this uses maps which is efficient for datasets
news_dataset = news_dataset.map(combine_news_text)
print("\n- - - sample data with full text- - -")
print(news_dataset[0])


Map:   0%|          | 0/5062 [00:00<?, ? examples/s]


- - - sample data with full text- - -
{'instruction': 'Dow drops 400 points, turns negative for the year as bank fears grow: Live updates', 'output': 'Regional banks led the broader market lower as contagion fears resurfaced.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction: Dow drops 400 points, turns negative for the year as bank fears grow: Live updates\n\n### Response: Regional banks led the broader market lower as contagion fears resurfaced.', 'full_text': 'Title:Dow drops 400 points, turns negative for the year as bank fears grow: Live updates\nDescriptionRegional banks led the broader market lower as contagion fears resurfaced.'}


In [11]:
#display the full text of first sample
print(news_dataset[0]["full_text"])

Title:Dow drops 400 points, turns negative for the year as bank fears grow: Live updates
DescriptionRegional banks led the broader market lower as contagion fears resurfaced.


In [12]:
display(news_dataset.select(range(10)).to_pandas())

,instruction,output,text,full_text
0,"Dow drops 400 points, turns negative for the y...",Regional banks led the broader market lower as...,Below is an instruction that describes a task....,"Title:Dow drops 400 points, turns negative for..."
1,Bitcoin drops with stocks as analyst warns of ...,Bitcoin dips as the U.S. banking crisis engulf...,Below is an instruction that describes a task....,Title:Bitcoin drops with stocks as analyst war...
2,Bitcoin Price Analysis: 29370 Tested After Su...,Bitcoin (BTC/USD) sought to add to recent gain...,Below is an instruction that describes a task....,Title:Bitcoin Price Analysis: 29370 Tested Af...
3,"Bitcoin Is 75% To Halving, Here's How Past Cyc...",The current Bitcoin cycle is now 75% on the wa...,Below is an instruction that describes a task....,"Title:Bitcoin Is 75% To Halving, Here's How Pa..."
4,"Iron Mountain FFO of $0.71 beats by $0.03, rev...",Iron Mountain press release (IRM): Q1 FFO of...,Below is an instruction that describes a task....,Title:Iron Mountain FFO of $0.71 beats by $0.0...
5,"Moderna GAAP EPS of $0.19 beats by $1.94, reve...",Moderna press release (MRNA): Q1 GAAP EPS of...,Below is an instruction that describes a task....,Title:Moderna GAAP EPS of $0.19 beats by $1.94...
6,Regeneron Pharmaceuticals Non-GAAP EPS of $10....,Regeneron Pharmaceuticals press release (REGN...,Below is an instruction that describes a task....,Title:Regeneron Pharmaceuticals Non-GAAP EPS o...
7,Apple earnings likely to show solid iPhone res...,Apple (AAPL) is set to report fiscal second-qu...,Below is an instruction that describes a task....,Title:Apple earnings likely to show solid iPho...
8,Rug Pulls and Meme Coins: Lessons from XIRTAM’...,The recent rug pull by XIRTAM raises concerns ...,Below is an instruction that describes a task....,Title:Rug Pulls and Meme Coins: Lessons from X...
9,Bitcoin Price Analysis: Stops Reached Below 28...,BTC/USD Stops Elected Below 28164: Sally Ho’s...,Below is an instruction that describes a task....,Title:Bitcoin Price Analysis: Stops Reached Be...


In [13]:
#accessing the 100th news in dataset
print(news_dataset[99]['full_text'])

Title:Stock futures fall as investors digest latest earnings, higher rates: Live updates
DescriptionInvestors kept their eyes on key earnings releases. Morgan Stanley and Tesla are due to report Wednesday.


**load and test Deepseek reasoning models**

In [14]:
print("Installing libraries...")
!pip install -q transformers accelerate bitsandbytes>=0.46.1 datasets torch gradio
print("successfully installed!")

Installing libraries...
successfully installed!


In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

#choosing deepseek model
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
print(f"loading model:{model_id}")
print(f"this may take a few minutes...")

# Define the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config=bnb_config, # Pass the quantization config
                                             device_map="auto",
                                             trust_remote_code=True,
                                             torch_dtype=torch.bfloat16 # Ensure model weights are loaded with bfloat16
)
model.eval()
print("Model loaded successfully in 4-bit")

loading model:deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
this may take a few minutes...


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully in 4-bit


In [16]:
#loading Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id,
                                        trust_remote_code = True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [17]:
#creating text generation pipeline
llm_pipelime = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    torch_dtype = "auto",
    device_map = "auto")
print("text-generation pipeline created!")

`torch_dtype` is deprecated! Use `dtype` instead!


text-generation pipeline created!


In [18]:
#building full formatted prompt
#test questions
test_question = "what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short"
#test_question = "Explain what are electric cars in simple terms.keep thinking short."
test_prompt = f"<|im_start|>user\n{test_question}\n<|im_end|>\n<|im_start|>assistant\n<think>\n"
print(f"Test prompt:\n{test_prompt}")

Test prompt:
<|im_start|>user
what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short
<|im_end|>
<|im_start|>assistant
<think>



In [19]:
#testing the model
#generate the text using pipeline
print("testing model")

outputs = llm_pipelime(test_prompt,
                       max_new_tokens =4000,
                       do_sample = True,
                       temperature = 0.7,
                       top_p = 0.9)
outputs

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


testing model


[{'generated_text': '<|im_start|>user\nwhat is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short\n<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let\'s see. The user wants to know the count of apples they have now. They initially had 5 apples. Then, they gave 2 to their mom and another 2 to their sister. So, they gave away a total of 4 apples. But wait, did they give away 2 to each or 2 in total? The way it\'s phrased, "gave 2 to my mom and 2 to my sister," probably means 2 each. So that\'s 4 apples given away. So, starting from 5, subtract 4. That leaves them with 1 apple. Yeah, that makes sense.\n</think>\n\nThe user initially had 5 apples. After giving 2 to their mom and 2 to their sister, they gave away a total of 4 apples. Therefore, the count of apples remaining is:\n\n5 (initial) - 4 (given away) = 1\n\n**Answer:** You have 1 apple now.'}]

In [20]:
#function to extract and format the reason and output
def format_model_output(output):
  reason_start = output.find("<think>") + len("<think>")
  reason_end = output.find("</think>")
  reason = output[reason_start:reason_end].strip()

  output_start = reason_end + len("</think>")
  model_output_content = output[output_start:].strip()

  reason = f"**Reason**:\n{reason}\n"
  output = f"**Output**:\n{output}\n"
  return reason, output

In [21]:
#extract and print generated text
full_output = outputs[0]['generated_text']
reason, output = format_model_output(full_output)
print_markdown(reason)
print_markdown(output)

**Reason**:
Okay, let's see. The user wants to know the count of apples they have now. They initially had 5 apples. Then, they gave 2 to their mom and another 2 to their sister. So, they gave away a total of 4 apples. But wait, did they give away 2 to each or 2 in total? The way it's phrased, "gave 2 to my mom and 2 to my sister," probably means 2 each. So that's 4 apples given away. So, starting from 5, subtract 4. That leaves them with 1 apple. Yeah, that makes sense.


**Output**:
<|im_start|>user
what is the count of apples i have now, if i initially had 5 appleswith me and then gave 2 to my mom and 2 to my sister.keep the thinking short
<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see. The user wants to know the count of apples they have now. They initially had 5 apples. Then, they gave 2 to their mom and another 2 to their sister. So, they gave away a total of 4 apples. But wait, did they give away 2 to each or 2 in total? The way it's phrased, "gave 2 to my mom and 2 to my sister," probably means 2 each. So that's 4 apples given away. So, starting from 5, subtract 4. That leaves them with 1 apple. Yeah, that makes sense.
</think>

The user initially had 5 apples. After giving 2 to their mom and 2 to their sister, they gave away a total of 4 apples. Therefore, the count of apples remaining is:

5 (initial) - 4 (given away) = 1

**Answer:** You have 1 apple now.


**PROMPTING FOR REASONING AND CLAASIFICATION**

This is the core of the project

Craft a prompt that tells the deepseek model

1. Analyze the provided financial news.

2. Think step-by-step from a financial perspective. why this news in positive negative or neutral.

3. output the reasoning

4. output the final classification.

using clear labels in the prompt helps the model structure its output.

**steps**

1. define a detailed prompt.

2. create a function analyze_news_sentiment that:

takes news text and the llm pipeline.

formats the prompt.

calls the pipeline to generate response.

parses the generated text to seperate the reasoning from final classification.

3. test the function on a few examples.

In [22]:
import re
def analyze_news_sentiment(news_text, llm_pipeline):
  """
  Analyzes the news sentiment using the LLM, providing reasoning and classification.

  Args:
    news_text (str): The combined title and description of the news.
    llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.
  Returns:
      str: A string containing the reasoning and classification.
  """

  #define the prompt template with specific instructions
  # Changed to use .format() to ensure news_text is evaluated only when the function is called
  question_template = """You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
{} """ # Placeholder for news_text

  test_question = question_template.format(news_text)

  #format the prompt according to Deepseeks Expected prompt with thinking tags
  test_prompt = f"<|im_start|>user\n{test_question}\n<|im_end|>\n<|im_start|>assistant\n<think>\n"

  #print the prompt for debuggig
  print(f"Test prompt:\n{test_prompt}")

  #Run the model interface with specific generation parameters.
  outputs = llm_pipeline(test_prompt,
                         max_new_tokens =4000,
                         do_sample = True,
                         temperature = 0.7,
                         top_p = 0.9)

  #extract the full generated text and parse into seperate reasoning and classifications
  #the format_model_output function likely separates the thinking process from final output
  full_output = outputs[0]["generated_text"]
  reason, output = format_model_output(full_output)

  return reason, output

In [23]:
#creating text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    torch_dtype = "auto",
    device_map = "auto")
print("text-generation pipeline created!")

text-generation pipeline created!


In [24]:
#Print a seperator line for clarity
print("\n"+ "="* 30 + "TESTING ANALYSIS" + "=" * 30 )
#select a few random indices from dataset to test the analysis function
random_indices = random.sample(range(len(news_dataset)), 3)

#iterate over each randomly selected index
for index in random_indices:
  #retrive the full text of the news items in current index
  sample_news = news_dataset[index]["full_text"]

  #analyze the sentiment of the full text
  reason, output = analyze_news_sentiment(sample_news, llm_pipeline)

  #print the results
  print(f"\n - - - Analysis result for index{index} - - -")

  #display the reasoning and output
  print_markdown(reason)
  print_markdown(output)

#printing seperate line for better readabilty results
print("-" * 60)



Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



==============================TESTING ANALYSIS==============================
Test prompt:
<|im_start|>user
You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
Title:Ghana Takes Steps to Operationalize Gold-for-Oil Scheme — Move Expected to Help Halt Cedi's Depreciation – Featured Bitcoin News
DescriptionThe gold-for-oil scheme is part of the Ghanaian government's plan to stop the further dwindling of the country's foreign exchange reserves.  
<|im_end|>
<|im_start|>assistant
<think>


 - - - Analysis result for index4221 - - -


**Reason**:
Alright, let me break down how I approached this. The user provided a news text and asked for specific outputs. I started by reading the news text to understand the main points. The title mentions Ghana taking steps to operationalize a gold-oil scheme, which is part of their plan to stop the depreciation of their foreign exchange reserves. The description reinforces this by stating it's part of their government's strategy.

First, I needed to extract the reasoning for the sentiment. The news text indicates that Ghana is working towards halting the depreciation of their currency, which suggests a positive outcome. So, the sentiment is positive.

Next, the sentiment is a choice between positive, negative, or neutral. Since the news is about halting depreciation, which is a positive outcome, the sentiment is positive.

Then, the tag should be a concise topic tag. The key elements here are Ghana, gold-oil scheme, and foreign exchange reserves. So, the tag is "Ghana, Gold-oil Scheme, Foreign Exchange Reserves."

I made sure to follow the user's instructions precisely, ensuring each output was correctly formatted and relevant to the provided news text.


**Output**:
<|im_start|>user
You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
Title:Ghana Takes Steps to Operationalize Gold-for-Oil Scheme — Move Expected to Help Halt Cedi's Depreciation – Featured Bitcoin News
DescriptionThe gold-for-oil scheme is part of the Ghanaian government's plan to stop the further dwindling of the country's foreign exchange reserves.  
<|im_end|>
<|im_start|>assistant
<think>
Alright, let me break down how I approached this. The user provided a news text and asked for specific outputs. I started by reading the news text to understand the main points. The title mentions Ghana taking steps to operationalize a gold-oil scheme, which is part of their plan to stop the depreciation of their foreign exchange reserves. The description reinforces this by stating it's part of their government's strategy.

First, I needed to extract the reasoning for the sentiment. The news text indicates that Ghana is working towards halting the depreciation of their currency, which suggests a positive outcome. So, the sentiment is positive.

Next, the sentiment is a choice between positive, negative, or neutral. Since the news is about halting depreciation, which is a positive outcome, the sentiment is positive.

Then, the tag should be a concise topic tag. The key elements here are Ghana, gold-oil scheme, and foreign exchange reserves. So, the tag is "Ghana, Gold-oil Scheme, Foreign Exchange Reserves."

I made sure to follow the user's instructions precisely, ensuring each output was correctly formatted and relevant to the provided news text.
</think>

1. POSITIVE
2. Positive
3. Ghana, Gold-oil Scheme, Foreign Exchange Reserves


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test prompt:
<|im_start|>user
You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
Title:4 signs the Bitcoin price rally could top out at $26K for now
DescriptionBitcoin price could revisit $19.5k based on a bearish technical pattern formation.  
<|im_end|>
<|im_start|>assistant
<think>


 - - - Analysis result for index1170 - - -


**Reason**:
Okay, let's see what I need to do here. The user provided a news text about Bitcoin and asked me to analyze it in three specific ways. First, I need to give a reasoning about the sentiment, which is the general mood or feeling expressed. Then, I have to determine whether the sentiment is positive, negative, or neutral. Finally, I need to create a tag that summarizes the main topic.

Looking at the news text, it says the Bitcoin price could rally to $26K. That's a positive statement because it indicates optimism about the market. So, the sentiment should be positive. The reasoning should explain why this is the case— Bitcoin's potential to recover after a bearish pattern.

For the tag, I need something concise that captures the essence of the article. Since it's about Bitcoin's potential rally, the tag should be clear and cover the main point. Something like "Bitcoin Potential Recovery" seems appropriate.

I should make sure not to add any extra text or explanations, just stick to the three requested items each on a new line, prefixed with the specified labels. Also, I need to ensure the reasoning is brief, as the user specified max points for that, so keeping it concise is key.

So, putting it all together, the response should have the reasoning explaining the positive sentiment, the sentiment being positive, and the tag summarizing the main topic. I think that's all covered in the initial response provided by the assistant, so I can safely follow that structure without making any mistakes.


**Output**:
<|im_start|>user
You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
Title:4 signs the Bitcoin price rally could top out at $26K for now
DescriptionBitcoin price could revisit $19.5k based on a bearish technical pattern formation.  
<|im_end|>
<|im_start|>assistant
<think>
Okay, let's see what I need to do here. The user provided a news text about Bitcoin and asked me to analyze it in three specific ways. First, I need to give a reasoning about the sentiment, which is the general mood or feeling expressed. Then, I have to determine whether the sentiment is positive, negative, or neutral. Finally, I need to create a tag that summarizes the main topic.

Looking at the news text, it says the Bitcoin price could rally to $26K. That's a positive statement because it indicates optimism about the market. So, the sentiment should be positive. The reasoning should explain why this is the case— Bitcoin's potential to recover after a bearish pattern.

For the tag, I need something concise that captures the essence of the article. Since it's about Bitcoin's potential rally, the tag should be clear and cover the main point. Something like "Bitcoin Potential Recovery" seems appropriate.

I should make sure not to add any extra text or explanations, just stick to the three requested items each on a new line, prefixed with the specified labels. Also, I need to ensure the reasoning is brief, as the user specified max points for that, so keeping it concise is key.

So, putting it all together, the response should have the reasoning explaining the positive sentiment, the sentiment being positive, and the tag summarizing the main topic. I think that's all covered in the initial response provided by the assistant, so I can safely follow that structure without making any mistakes.
</think>

1. REASONING: Bitcoin's potential to rally to $26K indicates optimism about the market, suggesting a positive sentiment.
2. SENTIMENT: Positive
3. TAG: Bitcoin Potential Recovery


------------------------------------------------------------


building gradio interface


In [25]:
import gradio as gr
import random
import re
from transformers import pipeline

In [ ]:
import gradio as gr
import random
import re
from transformers import pipeline
#gradio helper functions
def get_random_news():
  """Fetches and returns the full text of a random news item."""
  if not news_dataset:
    return "Error: News dataset not loaded."
  random_index = random.randint(0, len(news_dataset) - 1)
  news_text = news_dataset[random_index]["full_text"]
  print(f"Fetching news items at index{random_index}")
  return news_text

def perform_analysis(news_text):
  """Triggers analysis on provided news text."""
  if not news_text or news_text.startswith("Error"):
    return "Error: No news to analyze."
  # The `random_index` variable is not defined in this scope.
  # It seems it was expected to be available from `get_random_news` but isn't.
  # To proceed, we will remove this line. If `random_index` is crucial for debugging here,
  # it would need to be passed as an argument or fetched within this function.
  # print(f"Fetching news items at index{random_index}")
  reason, output = analyze_news_sentiment(news_text, llm_pipeline)
  return reason, output
#building gradio interface
with gr.Blocks() as demo:
  gr.Markdown("""
  #Deepseek Financial News Analyzer
  Fetches a random news item from dataset.
  Click 'Analyze News' to get sentiment classification and model reasoning.
  """)

  with gr.Row():
    btn_fetch = gr.Button("Fetch Random News")
    btn_analyze = gr.Button("Analyze News", variant = "primary")

  news_display = gr.Textbox(
      label = "News Text(Title & Description)",
      lines = 8,
      interactive = False,
      placeholder   = "Click 'Fetch Random News Item' to display news"
  )
  #creating a collapsible pannel
  with gr.Accordion("Model Reason", open =True):
    analysis_display = gr.Markdown()
  with gr.Accordion("Model Output", open = True):
    analysis_output = gr.Markdown()



  #event handlers

  btn_fetch.click(
      fn = get_random_news,
      inputs = [],
      outputs = [news_display]
  )
  btn_analyze.click(
      fn = perform_analysis,
      inputs = [news_display],
      outputs = [analysis_display, analysis_output]
  )

  #load initial news item when app start
  demo.load(
      fn = get_random_news,
      inputs = None,
      outputs = [news_display]
  )

  #launching gradio app
  print("launching gradio app...")
  demo.launch(share = True,
              debug = True,
              inline = True)
  print("gradio app launched!")

launching gradio app...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://425267a97ee332680f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Fetching news items at index1146
Fetching news items at index6


Both `max_new_tokens` (=4000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test prompt:
<|im_start|>user
You are a consise Financial News analyst. Analyze the provided news text. Output only the three requested items below, each on a new line, prefixed with the specified label.

1. REASONING: Brief financial reasoning(1-2 points max) for the sentiment.
2. SENTIMENT: Choose ONE: Positive, Negative, Neutral.
3. TAG: A consise topic tag(1-3 words).

DO NOT add any other text, greetings, or explanations.
News Text:
Title:Regeneron Pharmaceuticals Non-GAAP EPS of $10.09 beats by $0.53, revenue of $3.16B
DescriptionRegeneron Pharmaceuticals press release  (REGN): Q1  Non-GAAP EPS of $10.09  beats by $0.53. Revenue of $3.16B (+6.4% Y/Y)  beats by $160M. First quarter 2023 Dupixent global... 
<|im_end|>
<|im_start|>assistant
<think>

